In [1]:
import random
import os
import time
from typing import Any
from abc import ABC, abstractmethod
import random
import time
from typing import Any
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

class BaseModel(ABC):
    def __init__(self, model_name: str, **kwargs):
        self._name = model_name
        self.max_tokens = kwargs.get('max_tokens', 512)
        self.temperature = kwargs.get('temperature', 0.7)
        self.top_p = kwargs.get('top_p', 0.9)
        self.reasoning_effort = kwargs.get('reasoning_effort', None)
        self.n = kwargs.get('n', 1)
        self.input_tokens = 0
        self.output_tokens = 0

    @abstractmethod
    def generate(self, messages) -> str:
        pass

    @property
    def name(self) -> str:
        return self._name
    
    def from_text_to_tokens(self, text: str) -> list[int]:
        """Convert text to tokens."""
        raise NotImplementedError("This method should be implemented by subclasses.")
    
    def from_token_to_text(self, token: int) -> str:
        """Convert a token ID back to text."""
        raise NotImplementedError("This method should be implemented by subclasses.")

In [ ]:
from together import Together
import together

TogetherAIClient = Together(api_key="__YOUR_API_KEY__")


class TogetherAIModel(BaseModel):
    def __init__(self, model_name: str, **kwargs):
        super().__init__(model_name=model_name, **kwargs)

    def retry_with_exponential_backoff(  # type: ignore
        func,
        initial_delay: float = 1,
        exponential_base: float = 2,
        jitter: bool = True,
        max_retries: int = 5,
    ):
        """Retry a function with exponential backoff."""

        def wrapper(*args, **kwargs):  # type: ignore
            # Initialize variables
            num_retries = 0
            delay = initial_delay

            # Loop until a successful response or max_retries is hit or an exception is raised
            while True:
                try:
                    return func(*args, **kwargs)
                except together.error.InvalidRequestError as e:
                    raise e
                except Exception as e:
                    num_retries += 1
                    if num_retries < 7:
                        delay *= exponential_base * (1 + jitter * random.random())
                    print(f"#{num_retries} Error occurred: {e}.\n Retrying in {delay} seconds.")
                    # Sleep for the delay
                    time.sleep(delay)

        return wrapper

    @retry_with_exponential_backoff
    def generate(self, messages) -> str:
        """
        Chat completion using the chat/completions endpoint.
        Supports multi-modal inputs (text + images) for vision models.
        """
        response = TogetherAIClient.chat.completions.create(
            model=self.name,
            messages=messages,
            # max_tokens=self.max_tokens,
            max_new_tokens=1024,
            temperature=self.temperature,
            top_p=self.top_p,
            n=self.n,
        )
        
        usage = getattr(response, "usage", None)
        if usage:
            self.input_tokens += usage.prompt_tokens
            self.output_tokens += usage.completion_tokens
            print(f"Total input tokens: {self.input_tokens}, Total output tokens: {self.output_tokens}")

        # Raise OpenRouterError if we get invalid response to trigger retry
        if not response or not hasattr(response, 'choices') or not response.choices:
            raise ValueError("Zero response from Together API")

        predictions = [choice.message.content.strip(
        ) for choice in response.choices if choice.message.content.strip()]

        
        return predictions[0]

In [3]:
# model_name = "deepseek-ai/DeepSeek-R1-Distill-Llama-70B-free"
model_name ="lgai/exaone-deep-32b"
# model_name = "lgai/exaone-3-5-32b-instruct"

In [4]:

# agent = CodeGenerator("google/gemma-3-1b-it")
# agent = UnslothCodeGenerator("Qwen/Qwen2.5-Coder-3B-Instruct")
# agent = UnslothCodeGenerator("google/gemma-3-1b-it")
# agent = OpenRouterModel("openai/gpt-oss-20b:free")
agent = TogetherAIModel(model_name)

In [ ]:
import os
import time
import requests
from requests.exceptions import RequestException, Timeout
from google import genai
client = genai.Client()


API_KEY = "__YOUR_API_KEY__"
MODEL = "gemini-2.0-flash"
URL = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"

def gemini_prompt(prompt: str, max_retries: int = 3) -> str:
    """Send a prompt to Gemini API with retries + exponential backoff."""
    if not API_KEY:
        raise ValueError("Missing GEMINI_API_KEY. Please set it as an environment variable.")

    # headers = {
    #     "Content-Type": "application/json",
    #     "X-goog-api-key": API_KEY,
    # }

    # payload = {
    #     "contents": [
    #         {
    #             "parts": [{"text": prompt}]
    #         }
    #     ]
    # }

    # backoff = 5  # initial backoff in seconds
    # for attempt in range(1, max_retries + 1):
    #     try:
    #         # 10s connect timeout, 90s read timeout
    #         resp = requests.post(URL, headers=headers, json=payload, timeout=(120, 120))
    #         resp.raise_for_status()

    #         data = resp.json()
    #         return data["candidates"][0]["content"]["parts"][0]["text"]

    #     except (Timeout, RequestException) as e:
    #         if attempt == max_retries:
    #             raise  # re-raise if final attempt
    #         wait = backoff * (2 ** (attempt - 1))  # exponential backoff
    #         print(f"Attempt {attempt} failed: {e}. Retrying in {wait} seconds...")
    #         time.sleep(wait)

    # return "Failed after retries."
    response = client.models.generate_content(
    model=MODEL, contents=prompt)
    time.sleep(5)
    return response.text
model_name = MODEL.replace(".", "/")

In [6]:
import ast
import pandas as pd

def parse_tests(raw) -> list:
    raw = str(raw)
    x = ast.literal_eval(raw)
    if isinstance(x, str):
        x = ast.literal_eval(x)
    if not isinstance(x, (list, tuple)):
        raise ValueError("test_list parsed to non-list")
    return [str(t) for t in x]
    
def convert_csv_to_json(csv_file):
    df = pd.read_csv(csv_file, encoding='utf-8')
    df['test_list'] = df['test_list'].apply(parse_tests)
    
    if 'instruction_en' in df.columns:
        df['instruction_en'] = df['instruction_en'].str.replace(r'\s*Example:.*', '', regex=True)
    return df.to_dict(orient='records')

In [7]:
import signal

# Timeout handler
def _timeout_handler(signum, frame):
    raise TimeoutError("Execution timed out")

def evaluate_solution(solution_code: str, unit_tests: list[str], timeout_per_test: int = 5) -> int:
    # Clean solution code (optional: remove markdown fences)
    solution_code = solution_code.strip('` \n').replace('python\n', '').strip()
    
    # Prepare namespace
    namespace = {}
    
    # Execute solution code with timeout
    try:
        signal.signal(signal.SIGALRM, _timeout_handler)
        signal.alarm(timeout_per_test * len(unit_tests))  # total timeout for code + tests
        exec(solution_code, namespace)
        signal.alarm(0)
    except TimeoutError:
        print("⏱️ Timeout in solution code execution")
        return 0
    except Exception as e:
        print(f"❌ Error in solution code: {e}")
        return 0

    # Evaluate unit tests
    passed_count = 0
    for i, test_stmt in enumerate(unit_tests):
        try:
            signal.alarm(timeout_per_test)
            exec(test_stmt, namespace)
            signal.alarm(0)
            passed_count += 1
        except TimeoutError:
            print(f"⏱️ Test {i+1} timed out")
            signal.alarm(0)
        except AssertionError:
            print(f"❌ Test {i+1} failed: {test_stmt}")
            signal.alarm(0)
        except SystemExit as e:
            print(f"⚠️ SystemExit in test {i+1}: {e.code}")
            signal.alarm(0)
        except Exception as e:
            print(f"⚠️ Exception in test {i+1}: {e}")
            signal.alarm(0)

    return passed_count

In [8]:
def run_code(code: str):
    namespace = {}
    try:
        signal.signal(signal.SIGALRM, _timeout_handler)
        signal.alarm(60)  # total timeout for code + tests
        exec(code, namespace)
        signal.alarm(0)
    except TimeoutError:
        raise TimeoutError("Execution timed out")
    except AssertionError as e:
        raise AssertionError(f"Assertion failed: {e}")
    except SyntaxError as e:
        raise SyntaxError(f"Syntax error in code: {e}")
    except Exception as e:
        raise RuntimeError(f"Error while executing code: {repr(e)}")
    except SystemExit as e:
        raise RuntimeError(f"SystemExit occurred: {e.code}")
    except Exception as e:
        raise RuntimeError(f"Error while executing code: {repr(e)}")

In [9]:
def get_fix_instructions(error: Exception) -> str:
    if isinstance(error, TimeoutError):
        return (
            "⏱️ TimeoutError: The code took too long to finish running.\n"
            "- The program might be stuck in a loop or taking too long to process.\n"
            "- Check if there is a loop that doesn’t stop.\n"
            "- Test the code with smaller or simpler input data.\n"
            "- Make sure you're not doing unnecessary repeated calculations."
        )

    elif isinstance(error, AssertionError):
        return (
            f"🧪 AssertionError: {error}\n"
            "- The result of your code did not match what was expected.\n"
            "- Double-check the assertion conditions to ensure they are correct.\n"
            "- Carefully review your code logic to make sure it does what the test expects.\n"
            "- Make sure the values you're comparing are what you actually intended."
        )

    elif isinstance(error, SyntaxError):
        return (
            f"✏️ SyntaxError: {error}\n"
            "- There is a problem with how the code is written.\n"
            "- Check for missing colons `:`, parentheses `()`, or indentation.\n"
            "- Make sure strings are closed properly with matching quotes.\n"
            "- Review the line and nearby lines for typos or misplaced symbols."
        )

    elif isinstance(error, SystemExit):
        return (
            f"🚪 SystemExit: The program exited with code {error.code}.\n"
            "- The code called `exit()` or something that stops the program.\n"
            "- Only use exit calls if the program is supposed to stop.\n"
            "- If you don’t want the program to exit early, remove or comment out those lines."
        )

    elif isinstance(error, RuntimeError):
        return (
            f"🚨 RuntimeError: {repr(error)}\n"
            "- A general problem happened while the program was running.\n"
            "- Check the values being used in the part of the code that caused the error.\n"
            "- Make sure everything used has been defined correctly.\n"
            "- Ensure the logic and data flow make sense and follow the correct order."
        )

    else:
        return (
            f"❗ Unhandled Error: {repr(error)}\n"
            "- An unexpected issue occurred.\n"
            "- Review the error message to understand what part of the code is causing it.\n"
            "- Go over the code structure and logic step by step.\n"
            "- Make sure all variables and functions are used correctly."
        )


# Prompts

In [10]:
from tqdm import tqdm
import re
from IPython.display import clear_output
from pathlib import Path
import json

dev_set = convert_csv_to_json("test_v1_en_gemini.csv")

responses = []

SYSTEM_PROMPT = '''
You are a Python programming assistant. 

The user will provide a function stub where the original docstring is written in Bangla with a translated version and a unit test case.
Your task is to read the Bangla + English (Translated) docstring and the unit test case, understand the requirement, function parameters, return type, and complete the function implementation in Python. 
Your response must be in English, not Bangla, and must only contain valid Python code. 
Do not add explanations, comments, or extra text. Just return the code solution.
Your main task is to carefully read the Bangla + English (Translated) docstring and the unit test case and infer:
1. The expected number of parameter and their types
2. The expected return type
3. The correct implementation logic

Important guidelines:
1. The function signature is already provided in the instruction. Implement the function as specified.
2. Include a **main function** (using `def main:`) in your code that contains necessary unit tests or example calls to validate your function.
3. Do **not** call `main()` anywhere in your code. This will be executed externally.
4. Try to keep the code as simple as possible.
5. Your response should contain only one python block enclosed in a code block like:\n```python\n# your code here\n```.
'''

PROMPT_TEMPLATE = '''
{examples}

>> Your Task
> Instruction
```python
def {function_call}:
    """{instruction}"""
    """Translated: {instruction_en}"""
    """{docstring}"""
```

Now complete the python code for the function '{function_name}' and add a 'main' function with unit tests. You should use the 'check' function for unit tests, which is helpful for debugging. For example:

```python
def {function_call}:
    # Your code

def check(test_id, test_val, expected):
    assert test_val == expected, f"Test {{test_id}}: Expected {{expected}}, got {{test_val}}"

def main():
    {check_example}
    # Add more unit tests
```

'''

LAST_FAILED_ATTEMPT = '''
>> Last failed attempt

> Response:
{last_response}

> Error:
{last_error}

> Suggested Fix:
{fix_instructions}
'''

LAST_FAILED_CODE = '''
>> Last failed code

> Response:
{last_response}

> Error:
{last_error}

> Suggested Fix:
{fix_instructions}
'''

PYTHON_BLOCK_WARNING = '''
**Attention** Your last response doesn't contain any python block. Couldn't extract the code for testing.
Your response should contain only one python block enclosed in a code block like:\n```python\n# your code here\n```
Make sure to think less and give code in first 1000 tokens.
'''

LAST_SUCCESS_ATTEMPT = '''
>> Last successful attempt

> Code:
{last_response}

Your last code passed all the unit tests generated by you. Which means you are on the right track. Now try to handle corner cases.
'''

In [11]:
def get_arg_names_from_call(call_str: str):
    import ast
    tree = ast.parse(call_str, mode='eval')
    if not isinstance(tree.body, ast.Call):
        raise ValueError("Not a function call")
    arg_names = []
    for arg in tree.body.args:
        if isinstance(arg, ast.Name):
            arg_names.append(arg.id)
        else:
            arg_names.append(ast.unparse(arg))
    return arg_names

def parse_assert(assert_str: str):
    # Parse into AST
    tree = ast.parse(assert_str)

    # We assume the first statement is `assert`
    assert_node = tree.body[0]
    if not isinstance(assert_node, ast.Assert):
        raise ValueError("Not an assert statement")

    # Extract comparison (func(...) == expected)
    comp = assert_node.test
    if not isinstance(comp, ast.Compare):
        raise ValueError("Not a comparison in assert")

    # Left side: function call
    call = comp.left
    if not isinstance(call, ast.Call):
        raise ValueError("Left side is not a function call")

    func_name = call.func.id  # e.g. "max_chain_length"

    # Arguments of the function
    args_code = [ast.unparse(arg) for arg in call.args]

    # Expected value (right side of ==)
    expected_code = ast.unparse(comp.comparators[0])

    return func_name, args_code, expected_code

def assert_to_check(idx, assert_str: str) -> str:
    func_name, args_code, expected_code = parse_assert(assert_str)
    func_call = f"{func_name}({', '.join(args_code)})"
    return f"check({idx}, {func_call}, {expected_code})"
    
def to_docstring(s, func):
    func_name, args_code, expected_code = parse_assert(s)
    arg_names = get_arg_names_from_call(func)
    args = []
    for arg in args_code:
        arg_name = arg_names[len(args)]
        try:
            e_arg = eval(arg)
            args.append((arg_name, arg, type(e_arg)))
        except Exception as e:
            args.append((arg_name, arg, '<unknown type>'))

    try:
        expected = (expected_code, type(eval(expected_code)))
    except Exception as e:
        expected = (expected_code, '<unknown type>')
    
    template = """
    Args:
        {args}
        
    Returns:
        {returns}

    Example:
        >>> {example_function_call}
        {example_return}
    """
    
    arg_str = "\n        ".join(f"{name} ({type_}): Example: {example}" + (" (Try to infer the parameter type from example. If user-defined type needed, declare one.)" if type_ == '<unknown type>' else "") for name, example, type_ in args)

    return_str = f"{expected[1]}: Example: {expected[0]}"

    example_function_call = f"{func_name}({', '.join(args_code)})"

    example_return = expected_code
    
    return template.format(
        args=arg_str,
        returns=return_str,
        example_function_call=example_function_call,
        example_return=example_return
    )

In [12]:
def _get_function_call_and_name(item):
    function_head = item["instruction"].split("\n")[2].strip()
    match = re.search(r'def (.*?)\s*:', function_head)
    
    function_call = ""
    function_name = ""
    
    if match:
        function_call = match.group(1)
    else:
        function_call = function_head

    function_name, _, _ = parse_assert(item["test_list"][0])

    return function_call, function_name

In [13]:
def get_rag_examples(query, trial_set, num_examples=5):
    # Extract instruction_en from trial data
    # print(type(trial_set))
    trial_instructions = [item["instruction"].split("\n")[0].strip()+"\n"+item['instruction_en'].split("\n")[0].strip() for item in trial_set]
    vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
    all_instructions = trial_instructions + [query["instruction"].split("\n")[0].strip()+"\n"+query["instruction_en"].split("\n")[0].strip()]
    tfidf_matrix = vectorizer.fit_transform(all_instructions)
    query_vector = tfidf_matrix[-1]
    trial_vectors = tfidf_matrix[:-1]
    similarities = cosine_similarity(query_vector, trial_vectors).flatten()
    top_indices = np.argsort(similarities)[-num_examples:][::-1]
    return top_indices

In [14]:
EXAMPLE_TEMPLATE = '''
>> Example {idx}:
> Instruction
```python
def {function_call}:
    """{instruction}"""
    """Translated: {instruction_en}"""
    """{docstring}"""
```
> Solution
```python
{solution}

def check(test_id, test_val, expected):
    assert test_val == expected, f"Test {{test_id}}: Expected {{expected}}, got {{test_val}}"

def main():
    {test_main}
```
'''

def format_examples(trial_set, example_idx):
    examples = []
    count = 1
    for idx in example_idx:
        item = trial_set[idx]
        try:
            function_call, function_name = _get_function_call_and_name(item)
        except:
            print("Exception at: ", idx)
            raise
        instruction = item["instruction"].split("\n")[0].strip()
        instruction_en = item["instruction_en"].split("\n")[0].strip()
        docstring = to_docstring(item["test_list"][0], function_call)
        check_list = []
        for i, unit_test in enumerate(item["test_list"]):
            check_list.append(assert_to_check(i+1, unit_test))
    
        test_main = "\n    ".join(check_str for check_str in check_list)
        
        examples.append(EXAMPLE_TEMPLATE.format(
            function_call=function_call,
            instruction=instruction,
            instruction_en=instruction_en,
            docstring=docstring,
            solution=item["response"],
            idx=count,
            test_main=test_main
        ))
        count += 1

    EXAMPLES = "\n".join(example for example in examples)

    return EXAMPLES

In [15]:
trial_set = convert_csv_to_json("trial_with_en_v1.csv")

# print(format_examples(trial_set, [70]))

In [16]:
# import re
# import json
# dev = convert_csv_to_json("dev_en_gemini.csv")
# item = dev[0]
# print(item["test_list"][0])
# function_call = item["instruction"].split("\n")[2].strip()
# function_name = ""
# match = re.match(r"(\w+)\s*\(", function_call)
# if match:
#     function_name = match.group(1)
        
# default_messages = [
#             {"role": "system", "content": SYSTEM_PROMPT},
#         ]
# prompt = PROMPT_TEMPLATE.format(
#         instruction=item["instruction"].split("\n")[0].strip(),
#         instruction_en=item["instruction_en"].strip(),
#         function_call=function_call,
#         function_name=function_name,
#         examples=EXAMPLES,
#         last_failed_attempt="",
#         unit_test = item["test_list"][0]
#     )
# messages = default_messages + [{"role": "user", "content": prompt}]
# response = agent.generate(messages)
# print(response)

In [17]:
import time
count = 0
success = 0
total_test_count = 0
passed_test_count = 0
total = 0
max_attempt = 10

# Fixed tqdm bar at top
for item in tqdm(dev_set, desc="Generating", position=0):
    # open folder with task_id
    task_folder = Path(f"./results/{model_name}")
    task_folder.mkdir(parents=True, exist_ok=True)
    
    # create a submission.json file if doesn't exist
    if not task_folder.joinpath("submission.json").exists():
        with open(task_folder/"submission.json", "w", newline="", encoding="utf-8") as f:
            json.dump([], f, ensure_ascii=False)

    with open(task_folder/"submission.json", "r", encoding="utf-8") as f:
        submission_data = json.load(f)

    if any(submission["id"] == item["id"] for submission in submission_data):
        print(f"Skipping {item['id']} as it already exists in submission.json")
        matching_submission = next((submission for submission in submission_data if submission["id"] == item["id"]), None)
        flag = (matching_submission["score"] == 1.0)
        if flag:
            # success += flag
            # total += 1
            continue
        
    # prompt = item["instruction"].replace("Exammple", "Function call will be like following")
    last_error = None
    response = None
    fix_instructions = None
    attempt = 0
    history = []
    no_code = False
    last_success = False
    last_code = None
    record = {
        "id": item["id"],
        "response": ""
    }
    while True:
        try:
            function_call, function_name = _get_function_call_and_name(item)
        except Exception as e:
            print(e)
            continue

        check_example = "\n\t".join(
            (assert_to_check(i+1, ut) + " # Must pass test case") for i, ut in enumerate(item["test_list"])
        )
        # check_example = assert_to_check(1, item["test_list"][0]) + " # Must pass test case"

        example_idx = get_rag_examples(item, trial_set)
        print(f"Retrieved: {example_idx}")
        EXAMPLES = format_examples(trial_set, example_idx)
        
        prompt = PROMPT_TEMPLATE.format(
            instruction=item["instruction"].split("\n")[0].strip(),
            instruction_en=item["instruction_en"].split("\n")[0].strip(),
            function_call=function_call,
            function_name=function_name,
            examples=EXAMPLES,
            docstring=to_docstring(item["test_list"][0], function_call),
            check_example=check_example
        )

        # if last_error is not None:
        #     prompt += "\n" + LAST_FAILED_ATTEMPT.format(
        #         last_response=response[:1000]+"..." if no_code else response,
        #         last_error=last_error,
        #         fix_instructions=fix_instructions
        #     )
        # elif last_success:
        #     prompt += "\n" + LAST_SUCCESS_ATTEMPT.format(
        #         last_response=response,
        #     )

        # Python block handler
        if last_error is not None:
            prompt += "\n" + LAST_FAILED_CODE.format(
                last_response=last_code,
                last_error=last_error,
                fix_instructions=fix_instructions
            )
        if no_code:
            prompt += "\n" + PYTHON_BLOCK_WARNING
        elif last_success:
            prompt += "\n" + LAST_SUCCESS_ATTEMPT.format(
                last_response=last_code,
            )

        default_messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
        ]
        print(f"======================== {item['id']}.{attempt} =========================")
        
        # if no_code:
        # print(prompt)
        
        # messages = default_messages + [{"role": "user", "content": prompt}]
            
        # response = agent.generate(messages)
        
        messages = SYSTEM_PROMPT + prompt
        response = gemini_prompt(messages)
        
        #time.sleep(10)
        history.append({"role": "assistant", "content": response})
        
        # Extract Python code block
        pattern = re.compile(r"```python\s+([\s\S]*?)```", re.MULTILINE)
        match = pattern.search(response)
        record = {
            "id": item["id"],
            "response": ""
        }
        success_record = None
        if match:
            no_code = False
            code_inside = match.group(1)
            last_code = "```python\n" + code_inside + "\n```"

            # Python block handler
            response = last_code
            
            if re.search(rf"def\s+{function_name}\s*\(", code_inside):
                print(f"{function_name} function exists")
                pattern = r'if __name__ == ["\']__main__["\']:\n(?:[ \t]+.*\n?)*'
                clean_code = re.sub(pattern, '', code_inside, flags=re.MULTILINE)
                record = {
                    "id": item["id"],
                    "response": clean_code
                }    

                main_exist = re.search(r"def\s+main\s*\(\s*\)", code_inside)
                if main_exist:
                    print("Main function exists. Processing code.")
                    code_inside = clean_code + "\n\n# Call main function for testing\nmain()"

                print(code_inside)
                try:
                    run_code(code_inside)
                    last_error = None
                    success_record = record
                    # if last_success:
                    #     last_success = False
                    #     break
                    # else:
                    #     print("[Success] Handling corner cases.")
                    # last_success = True

                    if not main_exist:
                        print(f"No main function exists")
                        last_error = "No 'main' function found."
                        fix_instructions = "Add a 'main' function with unit tests. Please ensure your code is enclosed in a code block like:\n```python\n# your code here\n```"
                        attempt +=1
                        if attempt > max_attempt:
                            break
                    else:
                        break
                except AssertionError as e:
                    attempt += 1
                    last_error = e
                    fix_instructions = get_fix_instructions(last_error)
                    last_success = False
                    print(f"Error: {e}")
                    if attempt > max_attempt:
                        break
                except SyntaxError as e:
                    attempt += 1
                    last_error = e
                    fix_instructions = get_fix_instructions(last_error)
                    last_success = False
                    print(f"Error: {e}")
                    if attempt > max_attempt:
                        break
                except Exception as e:
                    attempt += 1
                    last_error = e
                    fix_instructions = get_fix_instructions(last_error)
                    last_success = False
                    print(f"Error: {e}")
                    if attempt > max_attempt:
                        break
            else:
                print(f"No {function_name} function exists")
                last_error = "No '"+function_name+"' function found."
                fix_instructions = "Rename the function to the provided function name.Please ensure your code is enclosed in a code block like:\n```python\n# your code here\n```\nMake sure to think less and give code in first 1000 tokens"
                attempt +=1
                last_success = False
                if attempt > max_attempt:
                        break
                continue  
        else:
            no_code = True
            last_success = False
            attempt +=1

            # Python block handler
            # last_error = "No Python code block found. Please ensure your code is enclosed in a code block like:\n```python\n# your code here\n```"
            # fix_instructions="Please ensure your code is enclosed in a code block like:\n```python\n# your code here\n```\nMake sure to think less and give code in first 1000 tokens"
            if attempt > max_attempt:
                break
            print("No python block")

        history.append({"role": "user", "content": f"{last_error}"})
        with open(task_folder/f"{item['id']}.json", "w", encoding="utf-8") as f:
            json.dump(history, f, ensure_ascii=False, indent=4)

    with open(task_folder/f"{item['id']}.json", "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=4)
        
    total += 1
    total_test_count += len(item["test_list"])
    if record is not None:
        if success_record is not None:
            record = success_record
        responses.append(record)
        count = evaluate_solution(record["response"], item["test_list"])
        success += (count == len(item["test_list"]))
        passed_test_count += count
        # add result to a submission.json
        index = next((i for i, submission in enumerate(submission_data) if submission["id"] == item["id"]), None)
        if index is None:
            submission_data.append({"id": item["id"], "response": record["response"], "score": count/len(item["test_list"])})
        else:
            submission_data[index] = {"id": item["id"], "response": record["response"], "score": count/len(item["test_list"])}
            
        with open(task_folder/"submission.json", "w", encoding="utf-8") as f:
            json.dump(submission_data, f, ensure_ascii=False, indent=4)

    # Clear previous logs and print updated stats
    # clear_output(wait=True)
    print(f"Task: {item['id']} -> Passed {count}/{len(item['test_list'])}")
    print(f"Complete: {success/total*100:.2f}%")
    print(f"Partial: {passed_test_count/total_test_count*100:.2f}%")

Generating:   0%|          | 0/500 [00:00<?, ?it/s]

Skipping 1 as it already exists in submission.json
Skipping 2 as it already exists in submission.json
Skipping 3 as it already exists in submission.json
Skipping 4 as it already exists in submission.json
Skipping 5 as it already exists in submission.json
Skipping 6 as it already exists in submission.json
Skipping 7 as it already exists in submission.json
Skipping 8 as it already exists in submission.json
Skipping 9 as it already exists in submission.json
Skipping 10 as it already exists in submission.json
Skipping 11 as it already exists in submission.json
Retrieved: [ 0 57 23 43 69]
======================== 11.0 =========================


Generating:   2%|▏         | 11/500 [00:08<06:01,  1.35it/s]

multiples_of_num function exists
Main function exists. Processing code.
def multiples_of_num(m,n):
    l = []
    for i in range(1, m):
        l.append(i*n)
    return l

def check(test_id, test_val, expected):
    assert test_val == expected, f"Test {test_id}: Expected {expected}, got {test_val}"

def main():
    check(1, multiples_of_num(4, 3), [3, 6, 9])
    check(2, multiples_of_num(5, 2), [2, 4, 6, 8])
    check(3, multiples_of_num(2, 5), [5])


# Call main function for testing
main()
❌ Test 1 failed: assert multiples_of_num(4,3)== [3,6,9,12]
Task: 11 -> Passed 0/1
Complete: 0.00%
Partial: 0.00%
Skipping 12 as it already exists in submission.json
Skipping 13 as it already exists in submission.json
Skipping 14 as it already exists in submission.json
Skipping 15 as it already exists in submission.json
Skipping 16 as it already exists in submission.json
Skipping 17 as it already exists in submission.json
Skipping 18 as it already exists in submission.json
Skipping 19 as it already e

Generating:   4%|▍         | 21/500 [01:53<49:57,  6.26s/it]

...

Generating:  91%|█████████ | 456/500 [35:07<02:26,  3.33s/it]

find_peak_util function exists
Main function exists. Processing code.
def find_peak_util(arr, n):
    if n == 1:
        return 0
    if arr[0] >= arr[1]:
        return 0
    if arr[n - 1] >= arr[n - 2]:
        return n - 1
    for i in range(1, n - 1):
        if arr[i] >= arr[i - 1] and arr[i] >= arr[i + 1]:
            return i

def check(test_id, test_val, expected):
    assert test_val == expected, f"Test {test_id}: Expected {expected}, got {test_val}"

def main():
    check(1, find_peak_util([1, 3, 20, 4, 1, 0], 6), 2)
    check(2, find_peak_util([10, 20, 15, 2, 23, 90, 67], 7), 1)
    check(3, find_peak_util([10, 20, 30, 40, 50], 5), 4)
    check(4, find_peak_util([50, 40, 30, 20, 10], 5), 0)
    check(5, find_peak_util([10], 1), 0)


# Call main function for testing
main()
Task: 456 -> Passed 1/1
Complete: 8.89%
Partial: 8.89%
Skipping 457 as it already exists in submission.json
Skipping 458 as it already exists in submission.json
Skipping 459 as it already exists in submissi

Generating:  95%|█████████▌| 475/500 [35:15<01:51,  4.45s/it]

is_palindrome function exists
Main function exists. Processing code.
def is_palindrome(arr,n):
    largest_palindrome = -1
    for num in arr:
        num_str = str(num)
        if num_str == num_str[::-1]:
            if num > largest_palindrome:
                largest_palindrome = num
    return largest_palindrome

def check(test_id, test_val, expected):
    assert test_val == expected, f"Test {test_id}: Expected {expected}, got {test_val}"

def main():
    check(1, is_palindrome([1, 232, 54545, 999991], 4), 54545)
    check(2, is_palindrome([121, 343, 565, 789], 4), 565)
    check(3, is_palindrome([12, 34, 56, 78], 4), -1)
    check(4, is_palindrome([11, 22, 33, 44], 4), 44)
    check(5, is_palindrome([1, 2, 3, 4],4), 4)


# Call main function for testing
main()
Task: 475 -> Passed 1/1
Complete: 10.87%
Partial: 10.87%
Retrieved: [ 2 62 26 24 36]


IndexError: list index out of range

In [ ]:
from pathlib import Path
import json
from tqdm import tqdm

dev_set = convert_csv_to_json("test_v1_en_gemini.csv")
task_folder = Path(f".results/{model_name}")
with open(task_folder/"submission.json", "r", encoding="utf-8") as f:
    submission_data = json.load(f)

count = 0
success = 0
total_test_count = 0
passed_test_count = 0
total = 0

for item in tqdm(dev_set, desc="Evaluating", position=0):
    # Find the json with submission["id"] == item["id"]
    matching_submission = next((submission for submission in submission_data if submission["id"] == item["id"]), None)
    if matching_submission:
        print(f"Evaluating {item['id']}...")
        count = evaluate_solution(matching_submission["response"], item["test_list"])
        if count == len(item["test_list"]):
            print(f"✅ All tests passed for {item['id']}")
        success += (count == len(item["test_list"]))
        total_test_count += len(item["test_list"])
        passed_test_count += count
        total += 1

print(f"Accuracy (Pass@1): {success/total*100:.2f}%")
print(f"Unit Test Success: {passed_test_count/total_test_count*100:.2f}%")

In [ ]:
print(f"Accuracy: {success/total*100:.2f}%")

import json
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"submission_{timestamp}.json"

with open(filename, 'w', encoding='utf-8') as f:
    json.dump(responses, f, ensure_ascii=False, indent=2)

print(f"Submission file: {filename}")